# PROC FCMP로 재사용 가능한 계리 요율산정 라이브러리 구축하기

## 요약

한 손해보험사가 순보험료, 비용/위험 부가, 제한적 변동 신뢰도 블렌딩, 할인 준비금 추정 등
핵심 요율산정 수식을 커스텀 함수와 다중 출력 서브루틴으로 **PROC FCMP**에 캡슐화한다.
컴파일된 라이브러리는 `CMPLIB=` 시스템 옵션을 통해 등록된 뒤, 합성 100건 계약 포트폴리오를
한 건씩 처리하는 DATA 스텝에서 호출된다. 이 노트북의 모든 산출 값 — 신뢰도 계수 `z`, 블렌딩된
순보험료, 부과 보험료, 현재가치 환산된 사고 준비금 — 은 인라인 산술이 아니라 이 컴파일된
루틴들에 의해 산출된다. 결과: 함의된 손해율은 농촌 60.5%, 교외 55.8%, 도시 63.6%로 — 모든
세그먼트에서 100%를 여유 있게 밑돌아, 부과된 보험료가 세그먼트별로 기대 손해를 충분히
충당하면서도 요율산정 단계가 깔끔하고 감사 가능하게 유지됨을 확인한다.


## 데이터 소스

| 데이터셋 | 행 수 | 설명 | 주요 변수 |
|---------|------|-------------|---------------|
| `work.portfolio` | 100 | `rand()`로 인라인 생성한 합성 유효계약 손해보험 포트폴리오 | `policy_id`, `region`(도시/교외/농촌), `years_insured`, `exposure`(차량-연), `claim_count`(Poisson), `incurred_loss`(감마 심도 x 건수), `class_pure_prem`(지역별 수동요율) |

DATA 스텝은 더 넓은 `policy_id` 범위를 순회하지만, 이 환경은 무인증 상태로 실행되므로 출력은
처음 **100개 관측치**로 제한된다 — 아래의 요율산정된 계약집은 그 100건을 반영한다.


# PROC FCMP로 재사용 가능한 계리 요율산정 라이브러리 구축하기

보험 계리사는 요율산정, 준비금 적립, 보고 작업 전반에서 동일한 계산을 반복한다. 손해액을
*순보험료*로 환산하고, *비용 및 위험 부가*를 적용해 부과 보험료에 도달하며, 개별 계약의
자체 경험을 *신뢰도*를 이용해 등급 요율과 블렌딩하고, 미래 현금흐름을 현재가치로 *할인*한다.
이 수식들을 DATA 스텝마다 다시 입력하면 복사-붙여넣기 오류를 부르고, 가정 변경을 고통스럽게
만든다.

**PROC FCMP**(SAS 함수 컴파일러)는 각 수식을 이름 붙은 함수나 서브루틴으로 한 번만 정의하고,
컴파일된 루틴을 라이브러리에 저장한 뒤, 다른 내장 SAS 함수처럼 호출할 수 있게 해준다. 이
노트북에서는 다음을 수행한다.

1. `PROC FCMP`로 소규모 계리 함수 라이브러리를 컴파일한다.
2. `CMPLIB=` 시스템 옵션으로 세션에 등록한다.
3. 합성 손해보험 포트폴리오를 생성한다.
4. 커스텀 함수와 다중 출력 서브루틴을 호출해, 단일 DATA 스텝에서 모든 계약을 요율산정한다.
5. 요율산정된 포트폴리오를 요약하고 해석한다.


## 1단계 — 합성 계약 포트폴리오 생성

유효 자동차 계약 장부를 시뮬레이션한다(무인증 상한 아래에서 아래에 요율산정되는 것은 처음
100건이다). 각 계약은 자체 순보험료(단위 노출당 기대 손해)를 가진 등급 지역에 속한다. 청구
건수는 노출로 스케일된 Poisson 과정을 따르고, 심도는 감마 분포를 따르므로 `incurred_loss`는
현실적인 복합(Poisson x 감마) 손해액이다. `years_insured`는 이후 신뢰도 가중치를 좌우한다.
`call streaminit`을 통한 고정 시드로 데이터의 재현성을 확보한다.


In [1]:
데이터 portfolio;
    호출 streaminit(20260531);
    길이 region $9;
    반복 policy_id = 10001 까지 12000;
        /* 요율 지역 배정 */
        u = rand('uniform');
        만약 u < 0.40 이면 반복; region = '도시';   base_pp = 820; lambda = 0.18; 종료;
        아니면 만약 u < 0.75 이면 반복; region = '교외'; base_pp = 540; lambda = 0.11; 종료;
        아니면 반복; region = '농촌';   base_pp = 360; lambda = 0.07; 종료;

        /* 가입 기간(년)과 노출(차량-연) */
        years_insured = 1 + rand('poisson', 3);
        exposure = round(0.5 + rand('gamma', 4) / 4, 0.01);

        /* 복합 청구 과정: Poisson 빈도, 감마 심도 */
        claim_count = rand('poisson', lambda * exposure);
        incurred_loss = 0;
        반복 c = 1 까지 claim_count;
            incurred_loss = incurred_loss + rand('gamma', 2.0) * 2500;
        종료;
        incurred_loss = round(incurred_loss, 0.01);

        /* 이 계약 노출에 대한 수동 등급 순보험료 */
        class_pure_prem = round(base_pp * exposure, 0.01);

        출력;
    종료;
    유지 policy_id region years_insured exposure claim_count
         incurred_loss class_pure_prem;
실행;

처리 인쇄 데이터=portfolio(obs=8) noobs;
    라벨 region='지역' years_insured='가입기간(년)' exposure='노출(차량-연)'
          claim_count='청구 건수' incurred_loss='발생손해액' class_pure_prem='등급 순보험료';
    제목 '시뮬레이션된 처음 8건의 계약';
실행;
제목;


                                                    시뮬레이션된 처음 8건의 계약                                                    

policy_id      지역            가입기간(년)            노출(차량-연)          청구 건수            발생손해액              등급 순보험료
    10001  농촌                      5                1.36              0                0                489.6
    10002  도시                      8                0.97              1          3432.28                795.4
    10003  도시                      2                1.53              2          7155.92               1254.6
    10004  교외                      9                 2.4              0                0                 1296
    10005  농촌                      5                0.99              0                0                356.4
    10006  교외                      1                0.83              0                0                448.2
    10007  농촌                      5                0.97              0                0                349.


NOTE: DATA portfolio

NOTE: Unlicensed mode - output limited to 100 observations.

NOTE: Wrote portfolio (100 rows, 7 columns).
NOTE: DATA elapsed:
  wall  0.42 seconds
  cpu   0.42 seconds
NOTE: PROC PRINT data=portfolio

NOTE: PROC PRINT completed: 8 observations printed, 7 variables


## 2단계 — 계리 함수 라이브러리 컴파일

이제 이 노트북의 핵심이다. `OUTLIB=work.actfuncs.pricing`을 지정한 `PROC FCMP`는 네 개의
함수와 하나의 서브루틴을 `work.actfuncs` 데이터셋의 `pricing` 패키지로 컴파일한다.

- **`pure_premium`** — 노출 단위당 관측 손해액(빈도 x 심도 결합).
- **`credibility_z`** — 제한적 변동 신뢰도 계수 `Z = sqrt(n / (n + k))`. 여기서 `n`은 계약의
  노출-연이고 `k`는 튜닝 상수이다.
- **`charged_premium`** — 비례 위험 부가와 고정 비용률을 손해원가에 적용해 실제로 부과하는
  보험료에 도달한다.
- **`pv_reserve`** — 미래 청구 지급액의 현재가치, `FV / (1+r)**t`, 사고 준비금 할인에 사용.
- **`blend_premium`**(서브루틴) — `OUTARGS`를 사용해 두 값을 동시에 반환한다: 신뢰도 가중
  순보험료와 사용된 신뢰도 계수. DATA 스텝은 한 번의 호출로 둘 다 받는다.

각 루틴은 `ENDSUB`로 닫히며, 서브루틴은 `OUTARGS`로 쓰기 가능한 인자를 지정한다.


In [2]:
처리 fcmp outlib=work.actfuncs.pricing;

    /* 관측 순보험료: 노출 단위당 손해원가 */
    function pure_premium(loss, exposure);
        만약 exposure <= 0 이면 RETURN(.);
        RETURN(loss / exposure);
    endsub;

    /* 제한적 변동 신뢰도: Z = sqrt(n / (n + k)) */
    function credibility_z(n_years, k);
        만약 n_years <= 0 이면 RETURN(0);
        RETURN(sqrt(n_years / (n_years + k)));
    endsub;

    /* 부과 보험료 = 위험 부가 + 비용을 반영해 그로스업한 손해원가 */
    function charged_premium(loss_cost, risk_load, expense_ratio);
        gross = loss_cost * (1 + risk_load);
        만약 expense_ratio >= 1 이면 RETURN(.);
        RETURN(gross / (1 - expense_ratio));
    endsub;

    /* r 이율로 t년 할인한 미래 청구 지급액의 현재가치 */
    function pv_reserve(future_value, r, t);
        RETURN(future_value / (1 + r) ** t);
    endsub;

    /* 신뢰도 블렌딩: 가중 순보험료와 사용된 Z를 함께 반환 */
    subroutine blend_premium(own_pp, class_pp, n_years, k, blended, z_used);
        outargs blended, z_used;
        z_used = credibility_z(n_years, k);
        blended = z_used * own_pp + (1 - z_used) * class_pp;
    endsub;

실행;



               The FCMP Procedure

  Output Library: work.actfuncs.pricing
  User-defined Function: pure_premium
  User-defined Function: credibility_z
  User-defined Function: charged_premium
  User-defined Function: pv_reserve
  User-defined Subroutine: blend_premium




NOTE: PROC FCMP outlib=work.actfuncs.pricing

NOTE: Function pure_premium stored to work.actfuncs.pricing.
NOTE: Function credibility_z stored to work.actfuncs.pricing.
NOTE: Function charged_premium stored to work.actfuncs.pricing.
NOTE: Function pv_reserve stored to work.actfuncs.pricing.
NOTE: Subroutine blend_premium stored to work.actfuncs.pricing.


## 3단계 — CMPLIB=로 라이브러리 등록

루틴을 컴파일하는 것만으로는 충분하지 않다. 이후의 DATA 스텝이나 프로시저가 인식하지 못하는
이름을 참조할 때 어디서 찾아야 하는지 SAS에 알려주어야 한다. `CMPLIB=` 시스템 옵션은(3단계
패키지 이름이 아니라) 컴파일된 코드를 담은 데이터셋을 가리킨다. 이 `OPTIONS` 문 이후로는 위의
모든 함수와 서브루틴을 이름으로 호출할 수 있다.


In [3]:
옵션 cmplib=work.actfuncs;



NOTE: Option CMPLIB changed to WORK.ACTFUNCS.


## 4단계 — 커스텀 함수로 포트폴리오 요율산정

이제 요율산정 DATA 스텝은 거의 산술이 없다 — 계리적 의도가 함수 이름에서 바로 읽힌다.
계약별로 다음을 수행한다.

1. `pure_premium`으로 계약 자신의 관측 순보험료를 계산한다.
2. `blend_premium` 서브루틴을 호출해 지역 등급 요율과 신뢰도 가중 블렌딩하고, `OUTARGS`를
   통해 블렌딩된 손해원가와 신뢰도 계수를 함께 회수한다.
3. `charged_premium`으로 블렌딩된 손해원가를 12% 위험 부가와 25% 비용률로 그로스업해 부과
   보험료에 도달한다.
4. 미결 사고 준비금을 계약 발생손해액의 35%로 추정하고, `pv_reserve`로 3년, 4%로 할인해
   현재가치를 구한다.

서브루틴의 출력 인자(`blended_pp`, `z`)는 `CALL` 이전에 반드시 초기화해야 한다. 준비금 PV는
계약별 실제 발생손해액에 따라 달라진다 — 무청구 계약은 준비금이 0이므로 `reserve_pv`도 0이다.


In [4]:
데이터 rated;
    설정 portfolio;

    /* 1. 계약 자신의 손해 경험을 순보험료로 */
    own_pp = round(pure_premium(incurred_loss, exposure), 0.01);

    /* 2. 자체 경험을 등급 요율과 신뢰도 가중 블렌딩.
          k = 4 노출-연을 사실상 완전 신뢰도로 사용. */
    blended_pp = .;
    z = .;
    호출 blend_premium(own_pp, class_pure_prem / exposure,
                       years_insured, 4, blended_pp, z);
    blended_pp = round(blended_pp, 0.01);
    z = round(z, 0.001);

    /* 3. 블렌딩된 손해원가(차량-연당)를 부과 보험료로 변환 */
    loss_cost = blended_pp * exposure;
    premium = round(charged_premium(loss_cost, 0.12, 0.25), 0.01);

    /* 4. 미결 사고 준비금 = 발생손해액의 35%, 3년 후 정산 예상;
          4%로 현재가치 할인. */
    case_reserve = round(0.35 * incurred_loss, 0.01);
    reserve_pv   = round(pv_reserve(case_reserve, 0.04, 3), 0.01);

    유지 policy_id region years_insured exposure incurred_loss
         own_pp class_pure_prem blended_pp z premium
         case_reserve reserve_pv;
실행;

처리 인쇄 데이터=rated(obs=10) noobs;
    라벨 region='지역' years_insured='가입기간(년)' exposure='노출(차량-연)'
          own_pp='자체 순보험료' blended_pp='블렌딩 순보험료' z='신뢰도(z)'
          premium='부과 보험료' reserve_pv='준비금 현재가치';
    제목 '요율산정된 포트폴리오 (처음 10건)';
    변수 policy_id region years_insured exposure own_pp
        blended_pp z premium reserve_pv;
실행;
제목;


                                                  요율산정된 포트폴리오 (처음 10건)                                                  

policy_id      지역            가입기간(년)            노출(차량-연)              자체 순보험료                블렌딩 순보험료        신뢰도(z)            부과 보험료                준비금 현재가치
    10001  농촌                      5                1.36                    0                   91.67         0.745            186.18                       0
    10002  도시                      8                0.97              3538.43                 3039.59         0.816           4402.95                 1067.95
    10003  도시                      2                1.53              4677.07                 3046.88         0.577           6961.51                 2226.55
    10004  교외                      9                 2.4                    0                   90.69         0.832            325.03                       0
    10005  농촌                      5                0.99                    0           


NOTE: DATA rated


NOTE: Read 100 rows from portfolio.
NOTE: Wrote rated (100 rows, 12 columns).
NOTE: DATA elapsed:
  wall  0.07 seconds
  cpu   0.07 seconds
NOTE: PROC PRINT data=rated

NOTE: PROC PRINT completed: 10 observations printed, 9 variables


## 5단계 — 요율산정된 포트폴리오 요약

동일한 컴파일 라이브러리를 통해 모든 계약이 요율산정되었으므로, 지역별로 포트폴리오를 롤업할
수 있다. 평균 부과 보험료, 평균 신뢰도 계수, 총 발생손해액, 총 현재가치 사고 준비금을
보고한 뒤, 세그먼트별로 함의된 손해율을 도출해 부과된 보험료가 기대 손해를 충당하는지
확인한다.


In [5]:
처리 평균 데이터=rated mean sum maxdec=2 nonobs;
    분류 region;
    변수 premium z incurred_loss reserve_pv;
    라벨 region='지역' premium='부과 보험료' z='신뢰도(z)'
          incurred_loss='발생손해액' reserve_pv='준비금 현재가치';
    제목 '지역별 포트폴리오 요약';
실행;

/* 함의된 손해율 = 발생손해액 / 부과 보험료, 지역별 현재가치 준비금과 함께 산출 */
처리 SQL;
    제목 '지역별 함의 손해율 및 준비금 PV';
    선택 region,
           count(*)                              AS n_policies,
           sum(incurred_loss)                    AS total_loss     형식=dollar12.2,
           sum(premium)                          AS total_premium  형식=dollar12.2,
           sum(incurred_loss) / sum(premium)     AS loss_ratio     형식=percent8.1,
           sum(reserve_pv)                       AS total_pv_reserve 형식=dollar12.2
    FROM rated
    GROUP 기준 region
    ORDER 기준 region;
QUIT;
제목;


                                                      지역별 포트폴리오 요약                                                      

                                                  The MEANS Procedure

                                      Analysis Variable : premium 부과 보험료

        지역                 Mean            Sum
        --------------------------------------
        교외               813.04       34147.74
        농촌               476.61       11438.62
        도시              1987.17       67563.93
        --------------------------------------

                                           Analysis Variable : z 신뢰도(z)

        지역                 Mean            Sum
        --------------------------------------
        교외                 0.68          28.36
        농촌                 0.71          17.14
        도시                 0.70          23.90
        --------------------------------------

                                   Analysis Variable : incurred_loss 발생손해액

        지역        


NOTE: PROC MEANS
NOTE: PROC MEANS statement used.
NOTE: PROC SQL 

NOTE: PROC SQL statement used.


## 결과 해석

요율산정 DATA 스텝은 단일 할인이나 신뢰도 수식을 인라인으로 풀어 쓰지 않는다 — 그저
`pure_premium`, `blend_premium`, `charged_premium`, `pv_reserve`를 호출할 뿐이다. 이것이
**PROC FCMP**의 성과이다: 계리적 가정이 단위 테스트, 버전 관리, 재사용이 가능한 하나의
컴파일된 라이브러리 안에 산다. 신뢰도 상수 `k`, 위험 부가, 비용률을 바꾸는 것은 모든 프로그램을
뒤지는 대신 라이브러리에서 한 번만 수정하면 된다.

요율산정된 계약집과 지역별 롤업을 읽어보면:

- **신뢰도(`z`)**는 제한적 변동 수식 `Z = sqrt(n / (n + k))`가 규정하는 그대로 `years_insured`에
  따라 증가한다. 처음 10건 중, 1년차 교외 계약(10006)은 `z = 0.447`, 2년차 도시 계약(10003)은
  `z = 0.577`, 9년차 교외 계약(10004)은 `z = 0.832`에 이른다. 경험이 얕은 계약은 지역 등급
  요율 쪽으로 당겨지고, 오래 가입한 계약은 자신의 손해 경험에 의존한다.
- **블렌딩 작동 방식:** 무청구 계약(계약집 대부분)은 `own_pp = 0`이므로, `blend_premium`은
  등급 요율에 `(1 - z)`를 곱한 값에 가까운 `blended_pp`를 반환한다 — 예를 들어 계약 10001
  (농촌, `z = 0.745`)은 `360`/차량-연 등급 요율 대비 `91.67`에 도달한다. 실제 손해가 발생한 두
  도시 계약, 10002와 10003은 대신 블렌딩된 손해원가를 자신의 높은 경험(`3039.59`, `3046.88`)
  쪽으로 끌어올린다.
- **부과 보험료**는 블렌딩된 손해원가보다 높은데, `charged_premium`이 12% 위험 부가를 더하고
  25% 비용률로 그로스업하기 때문이다. 이는 손해원가에 고정 `1.12 / 0.75 = 1.493` 승수를 적용한
  것과 같다. 평균 부과 보험료는 농촌 `476.61`, 교외 `813.04`, 도시 `1,987.17`이다.
- **할인된 준비금:** `pv_reserve`는 각 계약의 미결 사고 준비금(발생손해액의 35%)을 3년, 4%로
  할인하며, 이는 `0.889` 현재가치 계수에 해당한다. 무청구 계약은 `reserve_pv = 0`이고, 두
  도시 청구 계약은 각각 `1,067.95`와 `2,226.55`를 기여한다. 롤업하면, 계약집은 현재가치 준비금
  `$2,151.56`(농촌), `$5,932.52`(교외), `$13,364.13`(도시)을 보유한다.
- **함의된 손해율**은 농촌 60.5%, 교외 55.8%, 도시 63.6%에 이른다 — 모두 100%를 여유 있게
  밑돌아, 모든 세그먼트에서 부과 보험료가 기대 손해를 충당한다. 도시 세그먼트가 가장 뜨거운데,
  이는 두 건의 대형 시뮬레이션 손해에 의해 견인된 것이다 — 실제 요율 검토라면 수동요율을
  조정하기 전에 이 신호가 더 많은 사고연도에 걸쳐 지속되는지 검증할 것이다.

`blend_premium` 서브루틴은 또한 하나의 `CALL`에서 여러 결과를 반환하는 `OUTARGS` 관용구를
보여준다 — 블렌딩된 손해원가와 신뢰도 계수 `z`가 함께 돌아와, 별도의 함수 호출을 피하고
계약별 요율산정 로직을 간결하고 감사 가능하게 유지한다.
